In [2]:

import kagglehub


import os
import shutil
import sys
import subprocess


#Checks if you have the right version of python and are in the kaggle working directory.

print("Python version: ", sys.version)
print("Working directory: ", os.getcwd())

disk = shutil.disk_usage("/kaggle/working")


#Checks GPUs and CUDA version (BoltzGen requires CUDA 13)
result = subprocess.run(
    ["nvidia-smi"], capture_output = True,
    text = True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("GPU check failed.")
    print(result.stderr)


Python version:  3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Working directory:  /kaggle/working
Mon Oct 19 15:04:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                   

In [3]:
#BoltzGen needs to be installed in the kaggle environment

%pip install boltzgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 70.7 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached biotite-2.0.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
INFO: pip is looking at multiple versions of biotite to determine which version is compatible with other requirements. This could take a while.
  Using cached biotraj-1.2.2-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.1/357.1 kB 6.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 97.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━

In [4]:
%pip show boltzgen

Name: boltzgen
Version: 0.3.2
Summary: Protein design
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: biopython, biotite, cuequivariance_ops_cu12, cuequivariance_ops_torch_cu12, cuequivariance_torch, edit_distance, einops, einx, gemmi, huggingface_hub, hydra-core, hydride, logomaker, mashumaro, matplotlib, numba, numpy, nvidia-ml-py, pandas, pdbeccdutils, pydssp, pytorch-lightning, rdkit, scikit-learn, torch
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
import sys
import subprocess

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

result = subprocess.run(
    ["boltzgen", "--help"],
    capture_output=True,
    text=True
)
print("\nBoltzGen CLI exit code:", result.returncode)
print(result.stdout[:2000])
print(result.stderr[:1000])

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
PyTorch CUDA build: 12.8
CUDA available: True
GPU count: 2
GPU name: Tesla T4

BoltzGen CLI exit code: 0
usage: boltzgen [-h] [-v] {run,configure,execute,download,check,merge} ...

Boltzgen command line interface

positional arguments:
  {run,configure,execute,download,check,merge}
    run                 Run the binder design pipeline
    configure           Create configuration files for later execution
    execute             Run pipeline from pre-generated configuration files
    download            Download boltzgen model weights and supporting assets
    check               Validate design specification files
    merge               Combine finished pipeline outputs into a single
                        directory

options:
  -h, --help            show this help message and exit
  -v, --version         Print version and exit




In [3]:
#Copies the github repo of BoltzGen

%cd /kaggle/working
!git clone --depth 1 https://github.com/HannesStark/boltzgen.git

/kaggle/working
Cloning into 'boltzgen'...
remote: Enumerating objects: 375, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (331/331), done.
remote: Total 375 (delta 59), reused 211 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (375/375), 13.79 MiB | 22.77 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [7]:
from pathlib import Path

repo = Path("/kaggle/working/boltzgen")
examples = repo / "example"

# Lays out all of the examples to try
for path in sorted(examples.rglob("*.yaml"))[:40]:
    print(path.relative_to(repo))


example/binding_disordered_peptides/tpp4.yaml
example/binding_disordered_regions_of_proteins/hoxd13.yaml
example/binding_disordered_regions_of_proteins/npm1.yaml
example/binding_disordered_regions_of_proteins/nup98.yaml
example/cyclic_against_hiv_antibody_site/9d3d.yaml
example/cyclotide/3ivq.yaml
example/cyclotide/5wrd.yaml
example/cylcic_against_kras_with_specific_site/cyclicdesign.yaml
example/denovo_zinc_finger_against_dna/vanilla_protein.yaml
example/denovo_zinc_finger_against_dna/zinc_finger.yaml
example/design_spec_showcasing_all_functionalities.yaml
example/disulfide_peptide_with_betahairpin_conditioning/cropped_target.yaml
example/disulfide_peptide_with_betahairpin_conditioning/proximity_cropped_target.yaml
example/double_disulfide_peptide_against_specific_site/norepinephrine.yaml
example/fab_scaffolds/adalimumab.6cr1.yaml
example/fab_scaffolds/belimumab.5y9k.yaml
example/fab_scaffolds/crenezumab.5vzy.yaml
example/fab_scaffolds/dupilumab.6wgb.yaml
example/fab_scaffolds/golimum

In [7]:

## This is a test run of a protein, takes about 1 hour on kaggle's T4 x2 GPUs

!boltzgen run /kaggle/working/boltzgen/example/vanilla_protein/1g13prot.yaml \
  --output workbench/test_run \
  --protocol protein-anything \
  --num_designs 10 \
  --budget 2
# --num_designs is the number of intermediate designs. In practice you will want between 10,000 - 60,000
# --budget is how many designs should be in the final diversity optimized set


=== Configuring pipeline ===

mols.zip: downloading bytes: ██████████████████▌            |  234MB, 18.7MB/s  
mols.zip: reconstructing file:  17%|██▌            | 67.1MB /  391MB            
mols.zip: downloading bytes: █████████████████████████████▊ |  377MB, 32.7MB/s  
mols.zip: downloading bytes: ███████████████████████████████|  389MB, 34.4MB/s  
mols.zip: reconstructing file: 100%|███████████████|  391MB /  391MB, 36.0MB/s  
Using dataset artifact: /root/.cache/huggingface/hub/datasets--boltzgen--inference-data/snapshots/c3d36fd276e9caf098c75d4113c6d5eb320b1a4c/mols.zip
Creating output directory: workbench/test_run
************** Checking design spec: /kaggle/working/boltzgen/example/vanilla_protein/1g13prot.yaml **************
Total designed residues: 110
Design specification visualization is written to workbench/test_run/1g13prot.cif
******************************************************************************************************************
Using kernels: False [device c

In [8]:
#Checks the input (target) .cif file. If it compiles correctly put the output into Chimera/ PyMOL and check if it matches the desired target.

!boltzgen check /kaggle/input/datasets/vargab12345/trcm-dataset/TRCm.yaml


mols.zip: downloading bytes: ██████████████▋                |  186MB, 11.9MB/s  
mols.zip: downloading bytes: █████████████████████▏         |  268MB, 21.6MB/s  
mols.zip: downloading bytes: ███████████████████████████▍   |  347MB, 29.1MB/s  
mols.zip: downloading bytes: ███████████████████████████████|  389MB, 35.4MB/s  
mols.zip: reconstructing file: 100%|███████████████|  391MB /  391MB, 36.3MB/s  
Using dataset artifact: /root/.cache/huggingface/hub/datasets--boltzgen--inference-data/snapshots/c3d36fd276e9caf098c75d4113c6d5eb320b1a4c/mols.zip
************** Checking design spec: /kaggle/input/datasets/vargab12345/trcm-dataset/TRCm.yaml **************
Total designed residues: 98
Design specification visualization is written to TRCm.cif
*************************************************************************************************************


In [9]:
#The design run of 60 binding proteins (TCR mimics) binding to the pMHC. Takes about 4.5 hours on kaggle's T4 x2 GPUs.

!boltzgen run /kaggle/input/datasets/vargab12345/trcm-dataset/TRCm.yaml \
  --output workbench/real_run \
  --protocol protein-anything \
  --num_designs 60 \
  --budget 3



=== Configuring pipeline ===
Using dataset artifact: /root/.cache/huggingface/hub/datasets--boltzgen--inference-data/snapshots/c3d36fd276e9caf098c75d4113c6d5eb320b1a4c/mols.zip
Creating output directory: workbench/real_run
************** Checking design spec: /kaggle/input/datasets/vargab12345/trcm-dataset/TRCm.yaml **************
Total designed residues: 86
Design specification visualization is written to workbench/real_run/TRCm.cif
*************************************************************************************************************
Using kernels: False [device capability: (7, 5)]
Config overrides for protocol protein-anything: {}
Using 2 devices
Raw designs will be saved to: workbench/real_run/intermediate_designs
Using diffusion batch size: 1
Number of diffusion batches: 60

boltzgen1_diverse.ckpt: downloading bytes: ▌                | 61.5MB, 3.43MB/s  
boltzgen1_diverse.ckpt: reconstructing file:   3%| | 51.5MB / 1.93GB            
boltzgen1_diverse.ckpt: downloading byte